# 03 — Training: YOLOv8 Bib Detector

Trains a YOLOv8 model on the Roboflow-exported dataset to detect race bibs.

**Dataset:** `race-vision.v1i.yolov8` — 174 train / 22 val / 21 test, class: `bib`  
**Model:** YOLOv8n (nano) — fast to iterate on a small dataset; swap to `yolov8s` once baseline is solid  
**Device:** MPS (Apple Silicon) auto-detected, falls back to CPU

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Ultralytics version: {__import__('ultralytics').__version__}")

DATA_YAML = Path("../data/labeled/race-vision.v1i.yolov8/data.yaml").resolve()
RUNS_DIR  = Path("../models/runs").resolve()
RUNS_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_YAML.exists(), f"data.yaml not found at {DATA_YAML}"
print(f"data.yaml: {DATA_YAML}")
print(f"runs dir:  {RUNS_DIR}")

## Dataset sanity check

In [ ]:
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print(f"Classes ({cfg['nc']}): {cfg['names']}")

base = DATA_YAML.parent
for split in ("train", "val", "test"):
    img_dir = (base / cfg[split]).resolve()
    n = len(list(img_dir.glob("*.*")))
    print(f"  {split}: {n} images  ({img_dir})")

## Train

Key hyperparameters for a small dataset (174 images):
- `epochs=100` — small dataset trains fast; use early stopping via `patience`
- `imgsz=640` — YOLO default; bibs are small objects so don't go lower
- `batch=16` — fits comfortably in MPS memory
- `augment=True` — mosaic, flips, HSV jitter on by default in YOLOv8
- `patience=20` — stop early if val/mAP50 doesn't improve for 20 epochs

In [ ]:
model = YOLO("yolov8n.pt")  # downloads pretrained weights on first run

results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project=str(RUNS_DIR),
    name="bib-yolov8n",
    patience=20,
    exist_ok=True,
    verbose=True,
)

print(f"\nBest weights: {results.save_dir}/weights/best.pt")

## Training curves

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RUNS_DIR = Path("../models/runs")
results_csv = RUNS_DIR / "bib-yolov8n" / "results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

metrics = [
    ("train/box_loss",    "val/box_loss",        "Box loss"),
    ("train/cls_loss",    "val/cls_loss",         "Cls loss"),
    ("metrics/mAP50(B)",  "metrics/mAP50-95(B)", "mAP"),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col_a, col_b, title) in zip(axes, metrics):
    if col_a in df:
        ax.plot(df["epoch"], df[col_a], label=col_a.split("/")[-1])
    if col_b in df:
        ax.plot(df["epoch"], df[col_b], label=col_b.split("/")[-1])
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## Validate on val split

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUNS_DIR = Path("../models/runs")
best_weights = RUNS_DIR / "bib-yolov8n" / "weights" / "best.pt"
DATA_YAML = Path("../data/labeled/race-vision.v1i.yolov8/data.yaml").resolve()

import torch
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

model_best = YOLO(str(best_weights))
val_metrics = model_best.val(
    data=str(DATA_YAML),
    split="val",
    device=DEVICE,
    verbose=True,
)

print(f"\nmAP50:     {val_metrics.box.map50:.3f}")
print(f"mAP50-95:  {val_metrics.box.map:.3f}")
print(f"Precision: {val_metrics.box.mp:.3f}")
print(f"Recall:    {val_metrics.box.mr:.3f}")

## Spot-check: inference on test images

In [ ]:
import math
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
import matplotlib.pyplot as plt

RUNS_DIR     = Path("../models/runs")
DATASET_DIR  = Path("../data/labeled/race-vision.v1i.yolov8")
DEVICE       = "mps" if torch.backends.mps.is_available() else "cpu"

model_best  = YOLO(str(RUNS_DIR / "bib-yolov8n" / "weights" / "best.pt"))
test_images = sorted((DATASET_DIR / "test" / "images").glob("*.*"))[:9]
print(f"Test images found: {len(test_images)}")

preds = model_best.predict(
    source=[str(p) for p in test_images],
    conf=0.25,
    device=DEVICE,
    verbose=False,
)

COLS = 3
rows = math.ceil(len(preds) / COLS)
fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 5, rows * 4))
axes = np.array(axes).flatten()

for ax, (r, img_path) in zip(axes, zip(preds, test_images)):
    img = Image.fromarray(r.plot()[..., ::-1])
    ax.imshow(img)
    ax.set_title(img_path.name[:30], fontsize=8)
    ax.axis("off")

for ax in axes[len(preds):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("test_predictions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved test_predictions.png")

## Summary

| Artifact | Path |
|---|---|
| Best weights | `models/runs/bib-yolov8n/weights/best.pt` |
| Last weights | `models/runs/bib-yolov8n/weights/last.pt` |
| Training log | `models/runs/bib-yolov8n/results.csv` |
| Confusion matrix | `models/runs/bib-yolov8n/confusion_matrix.png` |

Next step: `04_evaluation.ipynb` — PR curve, confusion matrix deep-dive, failure mode analysis.